# ETGNN + Matched GAT Main C1 + C2 Benchmark (Validation Steps 37–40)

**Protocol:** Dated Frozen Protocol (v2.1) & Main C1+C2 Execution Plan  
**Target Hardware:** Google Colab GPU (T4 / V100 / A100) or High-RAM CPU  
**Scope:** Prospective confirmation benchmark across the **377 unique execution nodes in validation time steps 37–40**.

> [!IMPORTANT]
> **Methodological Integrity Guarantees:**
> 1. **Held-Out Validation Period:** Strict isolation to steps 37–40. Test period (steps 41–49) is completely absent.
> 2. **C1 Contrast:** Evaluates the 240-node sample with shared ETGNN-ranked and shared random deletions under a common target class (the frozen ETGNN predicted class). Labeled explicitly as **random-adjusted deletion-response difference** ($\delta_{\text{adj}}$) to adjust for baseline model perturbation noise, without claiming to isolate explanation fidelity from architectural differences.
> 3. **C2 Contrast:** Evaluates within-model explanation advantages ($A = \bar{D}_{\text{GE}} - \bar{D}_{\text{rnd}}$) using **each model's own independent GNNExplainer rankings and targets** on 4 primary and 4 degree-sensitivity caliper-matched pair registers. Matched-pair cluster bootstrap ($B=2{,}000$) aggregates repetitions within nodes.
> 4. **Incremental Resume & Backups:** Automatic incremental saving every 5 nodes with live Google Drive mirroring.

## 1. Inspect Hardware Environment
Check GPU availability and hardware specifications.

In [ ]:
!nvidia-smi
import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device Name:', torch.cuda.get_device_name(0))
    print('VRAM Total (GB):', round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))
else:
    print('Note: Running on CPU. GPU is recommended for speed.')

## 2. Mount Google Drive & Load Execution Bundle

Google Drive is mounted explicitly so that live incremental backups mirror automatically.
The bundle archive `ETGNN_Main_C1_C2_Execution_Bundle.zip` is **41.6 MB** (43,610,063 bytes).

In [ ]:
import os, shutil
from google.colab import drive

# Explicitly mount Google Drive for reliable transfer and live backup mirroring
print('Mounting Google Drive...')
drive.mount('/content/drive')

bundle_path = None
# Check if already uploaded directly to /content/
if os.path.exists('/content/ETGNN_Main_C1_C2_Execution_Bundle.zip'):
    if os.path.getsize('/content/ETGNN_Main_C1_C2_Execution_Bundle.zip') >= 43600063:
        bundle_path = '/content/ETGNN_Main_C1_C2_Execution_Bundle.zip'
        print('Found complete bundle in /content/.')

if bundle_path is None:
    drive_candidates = [
        '/content/drive/MyDrive/ETGNN_Main_C1_C2_Execution_Bundle.zip',
        '/content/drive/MyDrive/Colab Notebooks/ETGNN_Main_C1_C2_Execution_Bundle.zip',
    ]
    for c in drive_candidates:
        if os.path.exists(c):
            print(f'Found bundle in Drive: {c}')
            shutil.copy(c, '/content/ETGNN_Main_C1_C2_Execution_Bundle.zip')
            bundle_path = '/content/ETGNN_Main_C1_C2_Execution_Bundle.zip'
            break

if bundle_path is None:
    print('Bundle not found in Drive. Please upload via files.upload():')
    from google.colab import files
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith('.zip'):
            shutil.move(fn, '/content/ETGNN_Main_C1_C2_Execution_Bundle.zip')
            bundle_path = '/content/ETGNN_Main_C1_C2_Execution_Bundle.zip'
            print(f'Uploaded {fn} -> /content/ETGNN_Main_C1_C2_Execution_Bundle.zip')
            break

assert bundle_path and os.path.exists(bundle_path), 'CRITICAL: Execution bundle could not be loaded!'

## 3. Cryptographic Verification & Bundle Extraction

In [ ]:
import hashlib, zipfile

bundle_file = '/content/ETGNN_Main_C1_C2_Execution_Bundle.zip'
assert os.path.exists(bundle_file), f'Missing {bundle_file}!'

h = hashlib.sha256()
with open(bundle_file, 'rb') as f:
    while chunk := f.read(65536):
        h.update(chunk)
actual_sha = h.hexdigest()
actual_size = os.path.getsize(bundle_file)

print(f'Actual Size:   {actual_size:,} bytes')
print(f'Expected Size: 43,610,063 bytes')
print(f'Actual SHA256: {actual_sha}')
print(f'Expected SHA:  1a48608440287556bd5bf867902f07956def2c5a037ba075607a0d98ff016d3d')

assert actual_sha == '1a48608440287556bd5bf867902f07956def2c5a037ba075607a0d98ff016d3d', 'Cryptographic hash mismatch on bundle!'
print('Cryptographic verification PASSED. Extracting files...')

with zipfile.ZipFile(bundle_file, 'r') as zf:
    zf.extractall('/content')
print('Extraction complete. Workspace ready.')

## 4. Install Dependencies

In [ ]:
!pip install -q torch-geometric psutil
import torch_geometric, psutil
print('torch_geometric version:', torch_geometric.__version__)
print('psutil version:', psutil.__version__)

## 5. Development Plumbing Checks
1. Executes the complete benchmark pipeline on the **4 development benchmark nodes (steps 33–36)** to verify GPU execution, assertion logic, and memory stability in ~10 seconds.
2. Runs synthetic validation of Contrast 2 matched-pair cluster bootstrap across all 8 registers.

*Note:* Uses `subprocess.run(..., check=True)` to halt on any failure.

In [ ]:
import subprocess, sys

print('--- Running Dev-Check on 4 Development Nodes ---')
cmd_dev = [
    sys.executable,
    'src/evaluation/run_main_c1_c2_benchmark.py',
    '--dev-check',
    '--output-dir',
    '/content/experiments/dev_check_results'
]
subprocess.run(cmd_dev, check=True, cwd='/content')

print('\n--- Running Synthetic Contrast 2 Bootstrap Across All 8 Registers ---')
cmd_synth = [
    sys.executable,
    'src/evaluation/test_c2_all_registers_synthetic.py'
]
subprocess.run(cmd_synth, check=True, cwd='/content')
print('\nAll development plumbing assertions PASSED.')

## 6. Execute Main C1 + C2 Benchmark (Validation Steps 37–40)

Launches the authoritative main run on all **377 unique execution nodes**:
- Evaluates GNNExplainer (3 seeds) for ETGNN (C1 & C2) and Matched GAT (independent C2)
- Evaluates 30 shared random deletions per node
- Evaluates Degree and Occlusion comparators
- Asserts 100% raw/detailed consistency across all nodes and methods
- Saves checkpoints incrementally every 5 nodes
- Mirrors checkpoints and final statistical outputs live to Google Drive `/content/drive/MyDrive/ETGNN_Main_Backups`

In [ ]:
import os, subprocess, sys

drive_backup_dir = '/content/drive/MyDrive/ETGNN_Main_Backups'
os.makedirs(drive_backup_dir, exist_ok=True)
print(f'Live incremental backups will mirror to: {drive_backup_dir}')

cmd_main = [
    sys.executable,
    'src/evaluation/run_main_c1_c2_benchmark.py',
    '--config',
    'configs/main_c1_c2_run_config.json',
    '--output-dir',
    '/content/results/main_c1_c2_results',
    '--backup-dir',
    drive_backup_dir
]

# Using subprocess.run with check=True to halt on failure
subprocess.run(cmd_main, check=True, cwd='/content')

## 7. Display Benchmark Results & Statistical Summaries

In [ ]:
import pandas as pd, os

# 1. Contrast 1 Outcomes (Random-Adjusted Deletion-Response Difference)
c1_path = '/content/results/main_c1_c2_results/c1_statistical_results.csv'
if os.path.exists(c1_path):
    df_c1 = pd.read_csv(c1_path)
    print('=== CONTRAST 1 OUTCOMES (Random-Adjusted Deletion-Response Difference) ===')
    print(f'N = {len(df_c1)} nodes (steps 37–40)')
    print(f'Primary delta_adj:    Mean = {df_c1["delta_adj"].mean():.6f}, Std = {df_c1["delta_adj"].std():.6f}')
    print(f'Secondary delta_raw: Mean = {df_c1["delta_raw"].mean():.6f}, Std = {df_c1["delta_raw"].std():.6f}')
    print(df_c1.describe().to_string())

# 2. Contrast 2 Matched-Pair Cluster Bootstrap Across All 8 Registers
c2_path = '/content/results/main_c1_c2_results/c2_bootstrap_results.csv'
if os.path.exists(c2_path):
    df_c2 = pd.read_csv(c2_path)
    print('\n=== CONTRAST 2 OUTCOMES (Matched-Pair Cluster Bootstrap, B=2,000) ===')
    print(df_c2.to_string(index=False))

# 3. Individual Pair Differences Sample
pair_diff_path = '/content/results/main_c1_c2_results/c2_individual_pair_differences.csv'
if os.path.exists(pair_diff_path):
    df_pdiff = pd.read_csv(pair_diff_path)
    print(f'\nTotal matched pairs evaluated: {len(df_pdiff)}')
    print(df_pdiff.head(10).to_string())

## 8. Verified Results Packaging & Download
Gates packaging strictly on complete, verified outputs.

In [ ]:
import os, json, shutil
from google.colab import files

results_dir = '/content/results/main_c1_c2_results'
resume_file = f'{results_dir}/main_resume_state.json'
assert os.path.exists(resume_file), f'Missing {resume_file}!'

with open(resume_file, 'r', encoding='utf-8') as f:
    state = json.load(f)

assert state.get('status') == 'COMPLETED', f"Run not marked COMPLETED! Status: {state.get('status')}, completed: {state.get('completed_count')}/377"
assert state.get('completed_count') == 377, f"Incomplete run: {state.get('completed_count')}/377 nodes!"

# Verify required output files exist
for req_f in [
    'detailed_intervention_records.csv',
    'raw_benchmark_records.csv',
    'main_benchmark_summary.csv',
    'c1_statistical_results.csv',
    'c2_bootstrap_results.csv',
    'c2_individual_pair_differences.csv',
]:
    assert os.path.exists(f'{results_dir}/{req_f}'), f'Missing required output file: {req_f}'

print('All 377 nodes and statistical outputs verified. Creating results archive...')
archive_path = '/content/ETGNN_Main_C1_C2_Results.zip'
shutil.make_archive('/content/ETGNN_Main_C1_C2_Results', 'zip', results_dir)
print(f'Archive created: {archive_path} ({os.path.getsize(archive_path):,} bytes)')
files.download(archive_path)